### Importing necessary libraries

In [8]:
# Standard library
import os # Built-in module for interacting with the operating system (files, env vars, etc.)

# Third-party libraries
from dotenv import load_dotenv  # Loads environment variables from a .env file (e.g., API keys)
import requests  # Library for making HTTP requests (GET, POST, etc.)
from langdetect import detect, LangDetectException  # Detects the language of a given text
import gradio as gr  # Library for building simple web UIs (great for demos)

# LangChain core
from langchain_core.runnables import RunnableLambda  # Lets you wrap custom Python functions into LangChain pipelines
from langchain_core.embeddings import Embeddings  # Base class/interface for embedding models (used for custom implementations)
from langchain_core.documents import Document  # Standard document format used in LangChain (text + metadata)
from langchain_core.prompts import PromptTemplate  # Utility to define structured prompts with variables for LLMs
from langchain_core.output_parsers import StrOutputParser # Converts LLM output into plain string format (instead of structured objects)


# LangChain integrations
from langchain_huggingface import HuggingFaceEmbeddings   # Embedding model wrapper for Hugging Face models (used to convert text into vectors)
from langchain_text_splitters import RecursiveCharacterTextSplitter # Splits large text into smaller chunks while preserving context
from langchain_community.vectorstores import FAISS  # FAISS vector database for storing and searching embeddings locally (in-memory or disk)
from langchain_groq import ChatGroq  # Wrapper for Groq's LLM API (used to run models like Mixtral, Llama, etc.)

# Cookiejar Integration
from http.cookiejar import MozillaCookieJar  # Handles browser-style cookie storage (used if authentication/session is needed)

# YouTube
from youtube_transcript_api import YouTubeTranscriptApi  # Fetches subtitles/transcripts from YouTube videos
# Specific exceptions raised when transcript fetching fails
from youtube_transcript_api._errors import (
    TranscriptsDisabled,  # Video has no captions enabled
    NoTranscriptFound,    # No transcript available for requested language
    IpBlocked            # Request blocked (rate limiting or region restriction)
)

### Loading Environment Variables & Setting API Token

In [9]:
# Load environment variables from a .env file into the system environment
load_dotenv()
# Set Hugging Face API token in environment variables
# os.getenv("HF_TOKEN") reads the value of HF_TOKEN from the loaded .env file
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HF_TOKEN")

### LLM & Embedding Setup (E5 Multilingual + Groq)

In [10]:
# Initialize Groq LLM with a specific model
# temperature=0.2 → more deterministic, less creative responses
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.2)

# Hugging Face embedding model (multilingual, optimized for retrieval tasks)
BASE_EMBED_MODEL = "intfloat/multilingual-e5-large"

# Base embedding model instance
# normalize_embeddings=True → ensures cosine similarity works properly
hf_embedding = HuggingFaceEmbeddings(
    model_name = BASE_EMBED_MODEL,
    encode_kwargs = {"normalize_embeddings":True},
)

# Custom wrapper to enforce E5-specific input formatting
class E5MultilingualEmbeddings(Embeddings):
    """
    Thin wrapper around HuggingFaceEmbeddings that auto-applies
    the E5 prefix format required by multilingual-e5-* models:
      - Documents get  'passage: ' prefix
      - Queries get    'query: '   prefix
    Drop-in replacement anywhere LangChain expects an Embeddings object.
    """

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        # Add "passage:" prefix to each document
        prefixed = [f"passage: {t}" for t in texts]
        return hf_embedding.embed_documents(prefixed)
    
    def embed_query(self, text: str) -> list[float]:
        # Add "query:" prefix to user input
        return hf_embedding.embed_query(f"query: {text}")

# Instantiate the custom embedding class
embedding = E5MultilingualEmbeddings()

# Confirmation log
print(f"✅ Embedding model loaded: {BASE_EMBED_MODEL}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded: intfloat/multilingual-e5-large


### YouTube Video ID Extraction Function

In [11]:
def get_vid_id(url: str):
    """
    Extracts a YouTube video ID from a URL or returns the input if it's already an ID.
    Supports basic formats like:
    - https://www.youtube.com/watch?v=VIDEO_ID
    - VIDEO_ID (direct input)
    Returns:
        str | None → 11-character video ID or None if invalid
    """
    # Check if input looks like a URL
    if url[0:4] == "http":
        
        # Find where the video ID starts (after "v=")
        start = url.find("=") + 1
        
        # Find end of video ID (next parameter if exists)
        end = url.find("&")

        # If no additional parameters → take till end of string
        if end == -1:
            vid_id = url[start:]  
        else:
            vid_id = url[start:end]
    # Assume input is already a video ID
    else:
        vid_id = url

    # Validate YouTube video ID length (always 11 characters)    
    if len(vid_id) == 11:
        return vid_id
    else:
        return None

### YouTube Transcript Fetching + 🌍 Language Detection

In [12]:
def get_transcript(vid_id: str, cookie_file: str = "youtube_cookies.txt") -> tuple[str,str]:
    """
    Fetch transcript for any language, not just English.
    Strategy:
      1. List all available transcripts for the video.
      2. Prefer manually created captions over auto-generated ones.
      3. Among those, prefer English if available, else pick the first available language.
      4. Return (transcript_text, language_code).
    """

    # Create HTTP session to mimic a real browser (reduces blocking risk)
    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"        
            )
    })

    # Load cookies (useful for restricted videos or avoiding rate limits)
    cookie_jar = MozillaCookieJar(cookie_file)
    cookie_jar.load(ignore_discard=True, ignore_expires=True)
    session.cookies = cookie_jar

    # Initialize YouTube Transcript API with custom session
    yt_api = YouTubeTranscriptApi(http_client=session)

    # list all the available transcripts of a video
    transcript_list = yt_api.list(vid_id)

    # Separate meanual and auto-generated transcript
    manual = [t for t in transcript_list if not t.is_generated] 
    auto = [t for t in transcript_list if t.is_generated]

    # Helper: choose best transcript (prefer English if available)
    def pick_best(transcripts):
        if not transcripts:
            return None
        for t in transcripts:
            if t.language_code.startswith("en"):
                return t
        return transcripts[0]  # fallback to first available

    # Prefer manual captions first, then auto-generated
    chosen = pick_best(manual) or pick_best(auto)

    # If no transcript available → raise error
    if chosen is None:
        raise NoTranscriptFound(vid_id, [], [])
    
    # Fetch transcript content
    fetched = chosen.fetch()

    # Combine all subtitle chunks into a single string
    text =" ".join(chunk.text for chunk in fetched)

    return text, chosen.language_code


def detect_language(text: str) -> str:
    """
    Lightweight language detection via langdetect. 
    Return ISO 639-1 code (eg: 'en', "hi", 'fr').
    Falls back to "en" on failure.
    """
    try:
        return detect(text)
    except LangDetectException:
        return 'en'
        

### Build Retriever from Transcript (Chunking + FAISS)

In [13]:
def build_retriever(transcript: str, vid_id: str, transcript_lang: str = "en"):
    """
    Convert a transcript into a retriever for semantic search.

    Steps:
      1. Split transcript into overlapping chunks
      2. Wrap chunks into LangChain Document objects with metadata
      3. Convert chunks into embeddings
      4. Store embeddings in FAISS vector store
      5. Return a retriever for similarity-based search
    """
    # Split text into chunks with overlap to preserve context across boundaries
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=850,  # max characters per chunk
        chunk_overlap=150  # shared context between chunks
        )
    # Create Document objects with metadata for traceability    
    chunks: list[Document] = splitter.create_documents(
        texts = [transcript], 
        metadatas = [{
            "source" : "youtube",
            "video_id" : vid_id,
            "language" : transcript_lang,
        }]
        )
# Convert documents into embeddings and store in FAISS
    vector_store = FAISS.from_documents(
        documents=chunks, 
        embedding=embedding
        )
    
    # Return retriever interface for querying similar chunks
    return vector_store.as_retriever(
        search_type = 'similarity',   # cosine similarity search
        search_kwargs = {'k':4}   # return top 4 most relevant chunks
        )



### Output Parser Setup (String Parser)

In [14]:
# Converts LLM output into a plain Python string
parser = StrOutputParser()

### RAG Prompt + Chain Construction (LCEL Pipeline)

In [15]:

# Prompt template that constrains the LLM to use ONLY retrieved transcript context
rag_prompt = PromptTemplate(
        template="""\
            You are a helpful assistant that answers questions STRICTLY based on the YouTube \
            video transcript provided below.

            RULES:
            1. Use ONLY the information in the transcript context.
            2. If the answer is not present, reply: "I don't know — this topic isn't covered in the video."
            3. Be concise and factual.
            4. IMPORTANT: Respond in {answer_language}.

            --- TRANSCRIPT CONTEXT ---
            {context}
            --------------------------

            Question: {question}

            Answer:
        """,
        input_variables=['context', 'question', "answer_language"]
    )

def format_docs(docs: list[Document]) -> str:
    """Join retrieved chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

def build_rag_chain(retriever):
    """
    Build a fully LCEL-based RAG chain for the given retriever.

    Input:  {"question": str}  (the user's query)
    Output: str               (the answer)
    """

    def retrieve_and_format(inputs: dict) -> dict:
        """
        Step 1: Detect query language
        Step 2: Retrieve relevant chunks
        Step 3: Format them into context
        Step 4: Prepare prompt inputs
        """
        question = inputs["question"]

        # Detect user query language
        lang_code = detect_language(question)

        # Map ISO codes → readable language names for prompting
        lang_map = {
            "en": "English", "fr": "French", "es": "Spanish", "de": "German",
            "it": "Italian", "pt": "Portuguese", "nl": "Dutch", "ru": "Russian",
            "zh-cn": "Chinese", "zh-tw": "Chinese", "ja": "Japanese",
            "ko": "Korean", "ar": "Arabic", "hi": "Hindi", "tr": "Turkish",
            "pl": "Polish", "sv": "Swedish", "no": "Norwegian", "da": "Danish",
        }

        # Fallback if language not in map
        answer_language = lang_map.get(lang_code, f"{lang_code} (same language as the question)")

        # Retrieve relevant chunks from vector store
        docs = retriever.invoke(question)

        # Convert documents into a single context string
        context = format_docs(docs)

        return {
            "question" : question,
            "context" : context,
            "answer_language" : answer_language
        }
    

    # LCEL pipeline:
    # Input → retrieve → prompt → LLM → parse output
    chain = (
        RunnableLambda(retrieve_and_format) 
        | rag_prompt
        | llm
        | parser
    )

    return chain

### Full App Logic (Gradio + RAG + Summarization)

In [ ]:
# Global app state (shared across interactions)
# Stores current RAG pipeline, video ID, and transcript
pipeline_state = {
    "chain":None, 
    "vid_id":None, 
    "transcript":None
    }

def process_video(url):
    """
    Load video → extract transcript → build retriever → build RAG chain
    """
    vid_id = get_vid_id(url)
    if not vid_id:
        return (
            "⚠ Invalid YouTube URL or video ID.",
            gr.update(visible=False),  # hide QA section
        )

    try:
        # Fetch transcript + detected language
        transcript, transcript_lang = get_transcript(vid_id)

        # Build retriever from transcript
        retriever = build_retriever(transcript=transcript, vid_id=vid_id,transcript_lang=transcript_lang)

        # Build full RAG chain
        pipeline_state["chain"] = build_rag_chain(retriever)
        pipeline_state["vid_id"] = vid_id
        pipeline_state["transcript"] = transcript

        return (
            f"✅ Video loaded! ({vid_id}) — Transcript language: {transcript_lang}. Ask anything below.",
            gr.update(visible=True), # show QA section
            )
    
    # --- Error handling ---
    except FileNotFoundError: 
        return (
            "⚠ youtube_cookies.txt not found. Export cookies from your browser first.",
            gr.update(visible=False),
        )
    except IpBlocked:
        return (
            "⚠ IP blocked by YouTube. Re-export fresh cookies and try again.",
            gr.update(visible=False),
        )
    except TranscriptsDisabled:
        return (
            "⚠ Captions are disabled for this video.",
            gr.update(visible=False),
        )
    except NoTranscriptFound:
        return (
            "⚠ No transcript found for this video. The video may not have captions enabled.",
            gr.update(visible=False),
        )
    except Exception as e:
        return (
            f"⚠ Error loading video: {str(e)}",
            gr.update(visible=False),
        )



def answer_question(question: str) -> str:
    """
    Run RAG chain on user question
    """
    # Empty input check
    if not question.strip():
        return "⚠ Please type a question."
    
    # Ensure video is loaded
    if pipeline_state.get("chain") is None:
        return "⚠ No video loaded. Please load a video first."
    
    try:
        # Run RAG pipeline
        return pipeline_state["chain"].invoke({"question": question}), "" # also clears input box
    except Exception as e:
        err = str(e)

        # Basic API error handling (still string-based but acceptable)
        if "403" in err or "PermissionDenied" in err or "Access denied" in err:
            return (
                "⚠ Groq API blocked (403).\n"
                "Your IP or VPN is being blocked. Disable VPN → restart kernel → try again."
            )
        if "401" in err or "invalid_api_key" in err:
            return "⚠ Invalid Groq API key. Check your .env file."
        
        if "rate_limit" in err.lower():
            return "⚠ Groq rate limit hit. Wait a moment and try again."
        
        return f"⚠ Error: {err}"



def summarize():
    """
    Summarize transcript using semantic segmentation + map-reduce
    """

    from sklearn.metrics.pairwise import cosine_similarity

    transcript = pipeline_state.get("transcript")
    if not transcript:
        return "No transcript to summarize!"

    # --- Step 1: Split transcript into chunks ---
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )
    chunks = splitter.split_text(transcript)

    # --- Step 2: Generate embeddings ---
    embeddings_list = embedding.embed_documents(chunks)

    # --- Step 3: Segment chunks based on similarity ---
    def segment_chunks(chunks, embeddings, threshold=0.75):
        segments = []
        current_segment = [chunks[0]]

        for i in range(1, len(chunks)):
            sim = cosine_similarity(
                [embeddings[i - 1]],
                [embeddings[i]]
            )[0][0]
            # New segment if similarity drops
            if sim < threshold:
                segments.append(current_segment)
                current_segment = []

            current_segment.append(chunks[i])

        segments.append(current_segment)
        return segments

    segments = segment_chunks(chunks, embeddings_list)

    # --- Step 4: Summarize each segment ---
    map_prompt = PromptTemplate(
        template="""
Summarize this section in 2-3 bullet points.
Keep it concise and clear.

{text}
""",
        input_variables=["text"]
    )

    map_chain = map_prompt | llm | parser

    summaries = []
    for seg in segments:
        text = " ".join(seg)
        try:
            summary = map_chain.invoke({"text": text})
            summaries.append(summary)
        except Exception as e:
            summaries.append(f"⚠️ Error summarizing segment: {str(e)}")

    # --- Step 5: Combine all summaries ---
    reduce_prompt = PromptTemplate(
        template="""
You are an expert summary writer.

Combine the following section summaries into a final structured summary.

Rules:
- Cover all major topics
- 6-8 bullet points total
- Keep it concise
- Use emojis sparingly (optional)

{context}
""",
        input_variables=["context"]
    )

    reduce_chain = reduce_prompt | llm | parser

    try:
        final_summary = reduce_chain.invoke({
            "context": "\n".join(summaries)
        })
        return final_summary, ""

    except Exception as e:
        return f"⚠️ Error generating final summary: {str(e)}", ""


def reset_app():
    """
    Reset entire application state
    """
    pipeline_state["vid_id"] = None
    pipeline_state["chain"] = None
    pipeline_state["transcript"] = None
    return "", "", "", "App reset.", gr.update(visible=False)


# ---------------- UI ---------------- #
with gr.Blocks() as app:
    gr.Markdown("## ChatMyVideo_V3  😎")

    # Input for URL or video ID         
    url_input = gr.Textbox(label="🫠Enter YouTube URL / Video ID")
    
    # Setting Load + Reset Button
    with gr.Row():
        load_btn = gr.Button("🔃 Load Video")
        reset_btn = gr.Button("🔙 Reset")
    
    status_output = gr.Textbox(label="Status 🕵️‍♀️")
    
    # QA section (hidden initially)
    with gr.Column(visible=False) as qa_section:
        question_input = gr.Textbox(label="🙋 Ask a Question")
        with gr.Row():
            ask_btn = gr.Button("Ask🙋")
            summary_btn = gr.Button("Summarize😶‍🌫️") 
        
        answer_output = gr.Textbox(label="🤯  Answer")

    # --- Button bindings ---

    load_btn.click(
        process_video,
        inputs=url_input,
        outputs=[status_output, qa_section], 
        show_progress=True

    )

    ask_btn.click(
        answer_question,
        inputs=question_input,
        outputs=[answer_output,question_input], 
        show_progress=True
    )
    
    summary_btn.click(
        summarize,
        outputs=[answer_output, question_input], 
        show_progress=True            
    )

    reset_btn.click(
        reset_app,
        inputs=[],
        outputs=[url_input, question_input, answer_output, status_output, qa_section], 
        show_progress=True
    )

# Launch app
app.launch()

In [ ]:
# Global app state (shared across interactions)
# Stores current RAG pipeline, video ID, and transcript
pipeline_state = {"chain":None, "vid_id":None, "transcript":None}


def process_video(url):
    vid_id = get_vid_id(url)
    if not vid_id:
        return (
            "⚠ Invalid YouTube URL or video ID.",
            gr.update(visible=True),
            gr.update(visible=False),
        )

    try:
        transcript, transcript_lang = get_transcript(vid_id)

        retriever = build_retriever(transcript=transcript, vid_id=vid_id,transcript_lang=transcript_lang)

        
        pipeline_state["chain"] = build_rag_chain(retriever)
        pipeline_state["vid_id"] = vid_id
        pipeline_state["transcript"] = transcript

        return (
            f"✅ Video loaded! ({vid_id}) — Transcript language: {transcript_lang}. Ask anything below.",
            gr.update(visible=True),
            gr.update(visible=True),
            )
    
    except FileNotFoundError: 
        return (
            "⚠ youtube_cookies.txt not found. Export cookies from your browser first.",
            gr.update(visible=True),
            gr.update(visible=False),
        )
    except IpBlocked:
        return (
            "⚠ IP blocked by YouTube. Re-export fresh cookies and try again.",
            gr.update(visible=True),
            gr.update(visible=False),
        )
    except TranscriptsDisabled:
        return (
            "⚠ Captions are disabled for this video.",
            gr.update(visible=True),
            gr.update(visible=False),
        )
    except NoTranscriptFound:
        return (
            "⚠ No transcript found for this video. The video may not have captions enabled.",
            gr.update(visible=True),
            gr.update(visible=False),
        )
    except Exception as e:
        return (
            f"⚠ Error loading video: {str(e)}",
            gr.update(visible=True),
            gr.update(visible=False),
        )



def answer_question(question: str) -> str:
    if not question.strip():
        return "⚠ Please type a question."
    if pipeline_state.get("chain") is None:
        return "⚠ No video loaded. Please load a video first."
    
    try:
        return pipeline_state["chain"].invoke({"question": question}), ""
    except Exception as e:
        err = str(e)

        if "403" in err or "PermissionDenied" in err or "Access denied" in err:
            return (
                "⚠ Groq API blocked (403).\n"
                "Your IP or VPN is being blocked. Disable VPN → restart kernel → try again."
            )
        if "401" in err or "invalid_api_key" in err:
            return "⚠ Invalid Groq API key. Check your .env file."
        if "rate_limit" in err.lower():
            return "⚠ Groq rate limit hit. Wait a moment and try again."
        return f"⚠ Error: {err}"



def summarize():
    from sklearn.metrics.pairwise import cosine_similarity

    transcript = pipeline_state.get("transcript")
    if not transcript:
        return "No transcript to summarize!"

    # --- Step 1: Split transcript into chunks ---
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )
    chunks = splitter.split_text(transcript)

    # --- Step 2: Generate embeddings ---
    embeddings_list = embedding.embed_documents(chunks)

    # --- Step 3: Segment chunks based on similarity ---
    def segment_chunks(chunks, embeddings, threshold=0.75):
        segments = []
        current_segment = [chunks[0]]

        for i in range(1, len(chunks)):
            sim = cosine_similarity(
                [embeddings[i - 1]],
                [embeddings[i]]
            )[0][0]

            if sim < threshold:
                segments.append(current_segment)
                current_segment = []

            current_segment.append(chunks[i])

        segments.append(current_segment)
        return segments

    segments = segment_chunks(chunks, embeddings_list)

    # --- Step 4: Summarize each segment ---
    map_prompt = PromptTemplate(
        template="""
Summarize this section in 2-3 bullet points.
Keep it concise and clear.

{text}
""",
        input_variables=["text"]
    )

    map_chain = map_prompt | llm | parser

    summaries = []
    for seg in segments:
        text = " ".join(seg)
        try:
            summary = map_chain.invoke({"text": text})
            summaries.append(summary)
        except Exception as e:
            summaries.append(f"⚠️ Error summarizing segment: {str(e)}")

    # --- Step 5: Combine all summaries ---
    reduce_prompt = PromptTemplate(
        template="""
You are an expert summary writer.

Combine the following section summaries into a final structured summary.

Rules:
- Cover all major topics
- 6-8 bullet points total
- Keep it concise
- Use emojis sparingly (optional)

{context}
""",
        input_variables=["context"]
    )

    reduce_chain = reduce_prompt | llm | parser

    try:
        final_summary = reduce_chain.invoke({
            "context": "\n".join(summaries)
        })
        return final_summary, ""

    except Exception as e:
        return f"⚠️ Error generating final summary: {str(e)}", ""


def reset_app():
    pipeline_state["vid_id"] = None
    pipeline_state["chain"] = None
    pipeline_state["transcript"] = None
    return "", "", "", "App reset.", gr.update(visible=False)


with gr.Blocks() as app:
    gr.Markdown("## ChatMyVideo_V3  😎")

    url_input = gr.Textbox(label="🫠Enter YouTube URL / Video ID")
    
    with gr.Row():
        load_btn = gr.Button("🔃 Load Video")
        reset_btn = gr.Button("🔙 Reset")

    status_output = gr.Textbox(label="Status 🕵️‍♀️")

    with gr.Column(visible=False) as qa_section:
        question_input = gr.Textbox(label="🙋 Ask a Question")
        with gr.Row():
            ask_btn = gr.Button("Ask🙋")
            summary_btn = gr.Button("Summarize😶‍🌫️") 
        
        answer_output = gr.Textbox(label="🤯  Answer")

    load_btn.click(
        process_video,
        inputs=url_input,
        outputs=[status_output, qa_section], 
        show_progress=True

    )

    ask_btn.click(
        answer_question,
        inputs=question_input,
        outputs=[answer_output,question_input], 
        show_progress=True
    )
    
    summary_btn.click(
        summarize,
        outputs=[answer_output, question_input], 
        show_progress=True            
    )

    reset_btn.click(
        reset_app,
        inputs=[],
        outputs=[url_input, question_input, answer_output, status_output, qa_section], 
        show_progress=True
    )

app.launch()